### Import Library

In [71]:
# import library
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction import DictVectorizer

# import dataset
df = pd.read_csv("course_lead_scoring_2026.csv")

### Q1. Most frequent observation (mode) for the column **industry**

In [72]:
# Q1. most frequent observation (mode) for the column industry
print(df['industry'].mode())

print('Most frequent observation (mode) for column industry: technology')

0    technology
Name: industry, dtype: str
Most frequent observation (mode) for column industry: technology


### Q2. Biggest correlation

In [73]:
# Q2. two features that have the biggest correlation
corr_1 = df['interaction_count'].corr(df['lead_score'])
corr_2 = df['number_of_courses_viewed'].corr(df['lead_score'])
corr_3 = df['number_of_courses_viewed'].corr(df['interaction_count'])
corr_4 = df['annual_income'].corr(df['interaction_count'])

print(f'1. interaction_count & lead_score: {corr_1:.4f}')
print(f'2. number_of_courses_viewed & lead_score: {corr_2:.4f}')
print(f'3. number_of_courses_viewed & interaction_count: {corr_3:.4f}')
print(f'4. annual_income & interaction_count: {corr_4:.4f}')

data = {
    'interaction_count & lead_score': corr_1,
    'number_of_courses_viewed & lead_score': corr_2,
    'number_of_courses_viewed & interaction_count': corr_3,
    'annual_income & interaction_count': corr_4
}

max_data = max(data, key=data.get)

print(f'Two features that have the biggest correlation: {max_data}')

1. interaction_count & lead_score: 0.9287
2. number_of_courses_viewed & lead_score: 0.7694
3. number_of_courses_viewed & interaction_count: 0.7216
4. annual_income & interaction_count: 0.1064
Two features that have the biggest correlation: interaction_count & lead_score


### Q3. Biggest mutual information score

In [74]:
### Q3. Biggest mutual information score

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

categorical = ['lead_source', 'industry', 'employment_status', 'location']

def mutual_info_converted_score(series):
    return mutual_info_score(series, df_train['converted'])

mi = df_train[categorical].fillna('unknown').apply(mutual_info_converted_score)
print(mi.round(2).sort_values(ascending=False))

print('Variables has the biggest mutual information score: lead_source')

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64
Variables has the biggest mutual information score: lead_source


### Q4. Accuracy

In [75]:
# Q4. Accuracy
y_train = df_train['converted'].values
y_val = df_val['converted'].values

df_train = df_train.drop(columns=['converted'])
df_val = df_val.drop(columns=['converted'])

df_train = df_train.fillna(0)
df_val = df_val.fillna(0)

dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print(f'Accuracy: {round(accuracy, 2)}')

Accuracy: 0.65


### Q5. Feature has the smallest difference

In [76]:
# Q5. Feature has the smallest difference

orig_accuracy = accuracy

features_to_check = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

differences = {}

for feature in features_to_check:
    df_train_sub = df_train.drop(columns=[feature])
    df_val_sub = df_val.drop(columns=[feature])

    dv = DictVectorizer(sparse=False)
    
    train_dicts = df_train_sub.to_dict(orient='records')
    val_dicts = df_val_sub.to_dict(orient='records')
    
    X_train_sub = dv.fit_transform(train_dicts)
    X_val_sub = dv.transform(val_dicts)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train_sub, y_train)

    y_pred_sub = model.predict(X_val_sub)
    acc_sub = accuracy_score(y_val, y_pred_sub)

    diff = orig_accuracy - acc_sub
    differences[feature] = diff
    
    print(f'Without {feature:25} -> Accuracy: {acc_sub:.6f} | Difference: {diff:.6f}')

min_feature = min(differences, key=differences.get)
print(f'\nFeature has the smallest difference: {min_feature}')

Without lead_source               -> Accuracy: 0.642000 | Difference: 0.003000
Without number_of_courses_viewed  -> Accuracy: 0.643000 | Difference: 0.002000
Without interaction_count         -> Accuracy: 0.601000 | Difference: 0.044000

Feature has the smallest difference: number_of_courses_viewed


### Q6. Best accuracy on the validation set 

In [77]:
# Q6. Best accuracy on the validation set

c_values = [0.000001, 0.00001, 0.0001, 0.001]
scores = {}

for c in c_values:
    model = LogisticRegression(
        solver='liblinear', 
        C=c, 
        max_iter=1000, 
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    score = accuracy_score(y_val, y_pred)
    rounded_score = round(score, 3)
    
    scores[c] = rounded_score
    print(f'C = {c:<8} -> Accuracy: {score:.5f}')

best_c = max(scores, key=scores.get)

print(f'C, which yields the best accuracy: {best_c} with accuracy {scores[best_c]}')


C = 1e-06    -> Accuracy: 0.59800
C = 1e-05    -> Accuracy: 0.59800
C = 0.0001   -> Accuracy: 0.61300
C = 0.001    -> Accuracy: 0.64500
C, which yields the best accuracy: 0.001 with accuracy 0.645
